In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
def generate_embedding(text):
    return model.encode(text).tolist()

In [4]:
embedding = generate_embedding("What are the risk factors for this company?")
print(f"Length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Length: 384
First 5 values: [0.027860382571816444, 0.008152330294251442, -0.025170044973492622, -6.022820161888376e-05, 0.022478124126791954]


In [5]:
import numpy as np

chunks = [
    "This year saw significant strides in our understanding of XDR-47, a 'bug' we have not seen before.",
    "This division dedicated significant effort to studying various infection vectors in our distributed systems"
]

In [6]:
chunk_embeddings = [generate_embedding(chunk) for chunk in chunks]

for i, emb in enumerate(chunk_embeddings):
    print(f"Chunk {i+1} embedding length: {len(emb)}, first 3 values: {emb[:3]}")

Chunk 1 embedding length: 384, first 3 values: [-2.8779810236301273e-05, -0.03151053190231323, 0.004805091768503189]
Chunk 2 embedding length: 384, first 3 values: [-0.023848293349146843, -0.05704958736896515, -0.03182893991470337]


In [7]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def find_relevant_chunks(query, chunks, chunk_embeddings, top_k=1):
    query_embedding = generate_embedding(query)
    similarities = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]
    ranked = sorted(zip(similarities, chunks), reverse=True)
    return ranked[:top_k]

In [8]:
query = "What did the software engineering department do this year?"
results = find_relevant_chunks(query, chunks, chunk_embeddings)

for score, chunk in results:
    print(f"Similarity: {score:.3f}")
    print(f"Chunk: {chunk}")

Similarity: 0.278
Chunk: This year saw significant strides in our understanding of XDR-47, a 'bug' we have not seen before.


**Embedding function** now handles lists too (batch processing)
A VectorIndex class stores embeddings alongside original text
Search returns cosine distance (lower = more similar)

**Why Store Original Text**
Vector search returns numbers. You need the actual text to put in Claude's prompt. So store both together.

In [9]:
def generate_embedding(text):
    if isinstance(text, list):
        return [model.encode(t).tolist() for t in text]
    return model.encode(text).tolist()

In [10]:
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.metadata = []

    def add_vector(self, embedding, metadata):
        self.vectors.append(embedding)
        self.metadata.append(metadata)

    def search(self, query_embedding, top_k=2):
        distances = []
        for i, emb in enumerate(self.vectors):
            sim = cosine_similarity(query_embedding, emb)
            distances.append((1 - sim, self.metadata[i]))
        return sorted(distances, key=lambda x: x[0])[:top_k]

In [11]:
sample_text = """
## Section 1: Medical Research
This year saw significant strides in our understanding of XDR-47, a bug we have not seen before.

## Section 2: Software Engineering
This division dedicated significant effort to studying various infection vectors in our distributed systems.

## Methodology
We used a combination of qualitative and quantitative methods across all divisions.
"""

chunks = [c.strip() for c in sample_text.split("##") if c.strip()]

In [12]:
embeddings = generate_embedding(chunks)

store = VectorIndex()
for embedding, chunk in zip(embeddings, chunks):
    store.add_vector(embedding, {"content": chunk})

In [16]:
query = "What did the software engineering department do last year?"
query_embedding = generate_embedding(query)

results = store.search(query_embedding, top_k=2)

for distance, doc in results:
    print(f"Distance: {distance:.3f}")
    print(doc["content"][:200])
    print()

Distance: 0.621
Section 2: Software Engineering
This division dedicated significant effort to studying various infection vectors in our distributed systems.

Distance: 0.722
Methodology
We used a combination of qualitative and quantitative methods across all divisions.

